In [1]:
from eib import *
sns.set_context("paper")

In [2]:
data = {b: xr.open_dataset("../extended-ibtracs/extended-ibtracs_"+b+".nc") for b in BASINS}

/tmp/ipykernel_6569/347685557.py:1: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  data = {b: xr.open_dataset("../extended-ibtracs/extended-ibtracs_"+b+".nc") for b in BASINS}
/tmp/ipykernel_6569/347685557.py:1: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  data = {b: xr.open_dataset("../extended-ibtracs/extended-ibtracs_"+b+".nc") for b in BASINS}


In [3]:
# Distance between IBTrACS and sources
def haversine4(a,b,c,d):
    return haversine((a,b),(c,d))
haversine_vec = np.vectorize(haversine4)

In [4]:
# Compute dist2ib
for b in BASINS:
    data[b]["dist2ib"] = xr.apply_ufunc(haversine_vec, 
            data[b].sel(source = "IBTrACS").lat, 
            data[b].sel(source = "IBTrACS").lon, 
           data[b].lat, data[b].lon)

/home/users/sbourdin/.conda/envs/huracanpy/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:2625: RuntimeWarning: invalid value encountered in haversine4 (vectorized)
  outputs = ufunc(*args, out=...)


In [5]:
sns.set_palette("tab10")

df = pd.DataFrame()
for b in BASINS:
    eib = data[b]
    BINS = np.arange(0, 501, 20)

    # Track by track
    #fig, axs = plt.subplots()
    for ds in eib.source.values:
        if ds != "IBTrACS":
            D = eib.sel(source = ds)["dist2ib"].groupby(eib.track_id).mean()
            D_series = pd.Series(D.where(D.notnull(), drop = True).values).rename("D_track")
            D_df = pd.DataFrame(D_series).assign(source = ds, basin = b)
            df = pd.concat([df, D_df])
            #sns.histplot(data = D, 
            #             bins = BINS,
            #             fill = False, 
            #             #ec = PALETTE[ds],
            #             element = "step", 
            #             stat = "probability", 
            #             label = ds, 
            #             ax= axs)
            #axs.axvline(D.mean(), c = PALETTE[ds])
    #axs.legend()
    #axs.set_title(b)
    #axs.grid(axis = 'x')
    #axs.set_xlim(0,500)
    
    # TODO: Ajouter la moyenne pour chacun en dessous.

In [6]:
df

,D_track,source,basin
0,107.420060,SyCLoPS-ERA5,NA
1,90.244510,SyCLoPS-ERA5,NA
2,96.014648,SyCLoPS-ERA5,NA
3,58.086508,SyCLoPS-ERA5,NA
4,75.503433,SyCLoPS-ERA5,NA
...,...,...,...
1450,40.106237,TRACK-NCEP,WP
1451,64.190918,TRACK-NCEP,WP
1452,55.032649,TRACK-NCEP,WP
1453,50.569079,TRACK-NCEP,WP


In [ ]:
fig = plt.figure(figsize = cm2inch([9,15]))
ax = sns.boxplot(data = df[df.basin != "SA"], 
            x = "D_track", y = "source", 
            hue = "basin", #palette = PALETTE, 
           showfliers = False)
ax.set_xlim(0,500)
ax.set_ylabel("")
ax.set_xlabel("Track distance (km)")
ax.grid(axis = 'x')

plt.savefig("matching_distance.png", bbox_inches = "tight")
plt.savefig("matching_distance.pdf", bbox_inches = "tight")

In [ ]:
fig = plt.figure(figsize = cm2inch([9,10]))
ax = sns.boxplot(data = df[df.basin != "SA"], 
            x = "D_track", y = "basin", 
            hue = "source", palette = PALETTE, 
           showfliers = False)
ax.set_xlim(0,500)
ax.set_ylabel("")
ax.set_xlabel("Track distance in km")
ax.grid(axis = 'x')

ax.get_legend().remove()
fig.legend(ncol = 2, loc = "outside upper center", 
           bbox_to_anchor = (0.5, 0.0), frameon = True)
